In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [7]:
class Generator(nn.Module):
    def __init__(
        self,
        latent_dim: int,
        num_classes: int,
        embedding_dim: int,
        img_channels: int,
        features_g: int = 64,
    ):
        super().__init__()

        self.latent_dim = latent_dim
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim
        self.img_channels = img_channels
        self.features_g = features_g

        self.embedding = nn.Embedding(
            num_embeddings=num_classes, embedding_dim=embedding_dim
        )
        self.main = nn.Sequential(
            # Block 1: (N, latent_dim + embedding_dim, 1, 1) -> (N, features_g * 4, 4, 4)
            nn.ConvTranspose2d(
                latent_dim + embedding_dim, features_g * 4, 4, 1, 0, bias=False
            ),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            # Block 2: (N, features_g * 4, 4, 4) -> (N, features_g * 2, 8, 8)
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            # Block 3: (N, features_g * 2, 8, 8) -> (N, features_g, 16, 16)
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),
            # Block 4: (N, features_g, 16, 16) -> (N, img_channels, 32, 32)
            nn.ConvTranspose2d(features_g, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.ConvTranspose2d, nn.Conv2d, nn.Linear)):
                nn.init.normal_(m.weight.data, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight.data, 1.0, 0.02)
                nn.init.constant_(m.bias.data, 0)

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        y_embedded = self.embedding(y)

        combined_input = torch.cat([z, y_embedded], dim=1)

        x = combined_input.view(-1, self.latent_dim + self.embedding_dim, 1, 1)

        return self.main(x)


In [8]:
generator = Generator(100, 43, 50, 3, 128)

print(generator)

total_params = sum(param.numel() for param in generator.parameters())
print(f"{total_params=}")

test_z = torch.randn(2, 100)
test_y = torch.randint(0, 2, (2,))
generator(test_z, test_y).shape

Generator(
  (embedding): Embedding(43, 50)
  (main): Sequential(
    (0): ConvTranspose2d(150, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 3, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): Tanh()
  )
)
total_params=3860326


torch.Size([2, 3, 32, 32])

In [9]:
class Discriminator(nn.Module):
    def __init__(
        self,
        num_classes: int,
        embedding_dim: int,
        img_channels: int = 3,
        features_d: int = 64,
    ):
        super().__init__()

        self.num_classes = num_classes
        self.embedding_dim = embedding_dim
        self.img_channels = img_channels
        self.features_d = features_d

        self.embedding = nn.Embedding(
            num_embeddings=num_classes, embedding_dim=embedding_dim
        )

        self.main = nn.Sequential(
            # Block 1: (N, img_channels + embedding_dim, 32, 32) -> (N, features_d, 16, 16)
            nn.Conv2d(img_channels + embedding_dim, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Block 2: (N, features_d, 16, 16) -> (N, features_d * 2, 8, 8)
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # Block 3: (N, features_d * 2, 8, 8) -> (N, features_d * 4, 4, 4)
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # Block 4: (N, features_d * 4, 4, 4) -> (N, 1, 1, 1)
            nn.Conv2d(features_d * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.normal_(m.weight.data, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight.data, 1.0, 0.02)
                nn.init.constant_(m.bias.data, 0)

    def forward(self, img: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        y_embedded: torch.Tensor = self.embedding(y)

        y_embedded_spatial = y_embedded.unsqueeze(-1).unsqueeze(-1)
        y_embedded_spatial = y_embedded_spatial.expand(-1, -1, img.size(2), img.size(3))

        combined_input = torch.cat([img, y_embedded_spatial], dim=1)

        return self.main(combined_input).view(-1, 1)

In [10]:
discriminator = Discriminator(2, 50, 3, 64)

print(discriminator)

total_params = sum(param.numel() for param in discriminator.parameters())
print(f"{total_params=}")

test_img = torch.randn(2, 3, 32, 32)
test_y = torch.randint(0, 2, (2,))
discriminator(test_img, test_y).shape

Discriminator(
  (embedding): Embedding(2, 50)
  (main): Sequential(
    (0): Conv2d(53, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(256, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (9): Sigmoid()
  )
)
total_params=714596


torch.Size([2, 1])

In [11]:
import lightning as L
import wandb


class ConditionalDCGAN(L.LightningModule):
    def __init__(
        self,
        generator_params: dict,
        discriminator_params: dict,
        optimizer_params: dict,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.generator_params = generator_params
        self.discriminator_params = discriminator_params
        self.optimizer_params = optimizer_params

        self.generator = Generator(**generator_params)
        self.discriminator = Discriminator(**discriminator_params)

        self.criterion = nn.BCELoss()

        self.register_buffer("val_z", torch.randn(64, generator_params["latent_dim"]))
        self.register_buffer(
            "val_y",
            torch.randint(low=0, high=discriminator_params["num_classes"], size=(64,)),
        )

        self.automatic_optimization = False

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return self.generator(z, y)

    def training_step(self, batch, batch_idx):
        opt_g, opt_d = self.optimizers()

        real_images, labels = batch

        real_images = real_images.to(self.device)
        labels = labels.to(self.device)

        noise = torch.randn(
            real_images.size(0),
            self.generator_params["latent_dim"],
            device=self.device,
        )

        valid = torch.ones(real_images.size(0), 1, device=self.device)
        fake = torch.zeros(real_images.size(0), 1, device=self.device)

        # Train Generator
        opt_g.zero_grad()

        fake_images = self.generator(noise, labels)

        g_loss = self.criterion(self.discriminator(fake_images, labels), valid)

        self.log("train/generator_loss", g_loss, prog_bar=True)

        self.manual_backward(g_loss)
        opt_g.step()

        # Train Discriminator
        opt_d.zero_grad()

        real_loss = self.criterion(self.discriminator(real_images, labels), valid)
        fake_loss = self.criterion(
            self.discriminator(fake_images.detach(), labels), fake
        )

        d_loss = (real_loss + fake_loss) / 2

        self.log("train/discriminator_loss", d_loss, prog_bar=True)

        self.manual_backward(d_loss)
        opt_d.step()

    def on_train_epoch_end(self):
        self.generator.eval()

        with torch.inference_mode():
            sample_images = self.generator(self.val_z, self.val_y)
            grid = torchvision.utils.make_grid(sample_images, nrow=8, normalize=True)
            grid_image = wandb.Image(grid, caption="Generated Images")
            wandb.log({"generated_images": grid_image})

        self.generator.train()

    def configure_optimizers(self):
        generator_optimizer = torch.optim.Adam(
            self.generator.parameters(), **self.optimizer_params
        )
        discriminator_optimizer = torch.optim.Adam(
            self.discriminator.parameters(), **self.optimizer_params
        )
        return generator_optimizer, discriminator_optimizer


In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import random_split, DataLoader

data_root_folder = "data"
image_size = 32

batch_size = 128

data_transform = transforms.Compose(
    [
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
)

full_dataset = ImageFolder(root=data_root_folder, transform=data_transform)
# num_classes = len(full_dataset.classes)1ge(0, len(full_dataset), 64))


train_ratio = 0.8
val_ratio = 1.0 - train_ratio

train_size = int(train_ratio * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])


train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True
)

In [13]:
from lightning import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger


wandb_logger = WandbLogger(project="trafic-signs-generation")

num_classes = len(full_dataset.classes)

generator_params = {
    "latent_dim": 128,
    "num_classes": num_classes,
    "embedding_dim": 64,
    "img_channels": 3,
    "features_g": 96,
}

discriminator_params = {
    "num_classes": num_classes,
    "embedding_dim": 64,
    "img_channels": 3,
    "features_d": 96,
}
optimizer_params = {"lr": 0.0002, "betas": (0.5, 0.999)}

model = ConditionalDCGAN(
    generator_params=generator_params,
    discriminator_params=discriminator_params,
    optimizer_params=optimizer_params,
)

checkpoint_cb = ModelCheckpoint(
    save_on_train_epoch_end=True,
    save_top_k=-1,
    filename="model-{epoch:02d}",
)


trainer = Trainer(
    max_epochs=100,
    accelerator="mps",
    callbacks=[checkpoint_cb],
    log_every_n_steps=10,
    logger=wandb_logger,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

wandb.finish()


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/daniel/Studia/Semestr_6/SSNE/projekt_4/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
wandb: Currently logged in as: danielmachniak423 (danielmachniak423-politechnika-warszawska) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



  | Name          | Type          | Params | Mode 
--------------------------------------------------------
0 | generator     | Generator     | 2.7 M  | train
1 | discriminator | Discriminator | 1.6 M  | train
2 | criterion     | BCELoss       | 0      | train
--------------------------------------------------------
4.3 M     Trainable params
0         Non-trainable params
4.3 M     Total params
17.002    Total estimated model params size (MB)
28        Modules in train mode
0         Modules in eval mode
/Users/daniel/Studia/Semestr_6/SSNE/projekt_4/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/daniel/Studia/Semestr_6/SSNE/projekt_4/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' arg

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇█
train/discriminator_loss,▃▃▃▄▄▄▃▃▃▃▃▃▅▄▃▃▃▄▃▃▃▃▃▂▂▂▃▂▂▂▂▂█▂▂▁▅▃▁▁
train/generator_loss,▅▃▄▂▄▃▂▂▁▃▅▅▂▂▄▃▅▂▃▄▂▄▂▂▇█▄▃▃▅▃▃▃▅▄▇▅▆█▄
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇█
epoch,99
train/discriminator_loss,0.58593
train/generator_loss,4.19518
trainer/global_step,24599
